## CELL 1 — Imports & environment

In [1]:
%pip install -q rapidfuzz jellyfish awswrangler

import sys

if sys.version_info < (3, 8):
    raise RuntimeError(f"Python 3.8+ required. You have {sys.version}.")

_import_errors = []
try:
    import re, gc, json, time, uuid
    import numpy as np
    import pandas as pd
except ImportError as e:
    raise RuntimeError(f"Missing core package: {e}")

try:
    import psutil
except ImportError:
    psutil = None

try:
    import jellyfish
except ImportError:
    _import_errors.append("jellyfish")
try:
    import rapidfuzz
    from rapidfuzz import fuzz
except ImportError:
    _import_errors.append("rapidfuzz")

if _import_errors:
    raise RuntimeError(
        f"Missing packages: {_import_errors}. "
        f"Run: %pip install -q {' '.join(_import_errors)}"
    )

from datetime import datetime, timezone

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

def mem_gb():
    """Current process RSS in GB."""
    if psutil is None:
        return None
    return psutil.Process().memory_info().rss / (1024**3)

def format_mem_gb():
    """Formatted memory usage for logs; works when psutil is unavailable."""
    memory_gb = mem_gb()
    return f"{memory_gb:.1f}GB" if memory_gb is not None else "unavailable"

print("=" * 50)
print("Environment Check:")
print("=" * 50)
print(f"python:    {sys.version.split()[0]}")
print(f"pandas:    {pd.__version__}")
print(f"numpy:     {np.__version__}")
print(f"rapidfuzz: {rapidfuzz.__version__}")
print(f"jellyfish: installed")
print(f"Memory:    {format_mem_gb()}")
print("=" * 50)
print("\nReady to use!")

Note: you may need to restart the kernel to use updated packages.


Environment Check:
python:    3.12.13
pandas:    2.3.3
numpy:     1.26.4
rapidfuzz: 3.14.5
jellyfish: installed
Memory:    0.2 GB

Ready to use!


## CELL 2 — CONFIG (the only block business tunes)


In [2]:
#with error handling

FIELD_WEIGHTS = {
    "surname":   15,
    "givenname": 15,
    "dob":       15,
    "address":   15,
    "email":     10,
    "mobileno":  10,   # compared via the phone pool (mobile + home + office)
    "postal":    10,
    "homeno":     5,
    "officeno":   5,
}

# ---- Classification thresholds ----
THRESHOLD_MERGE      = 90   # >= MERGE (needs strong id + supporting field, no conflict)
THRESHOLD_REVIEW_LOW = 70   # 70..89 -> EYEBALL ;  < 70 -> UNIQUE

# ---- Fuzzy cut-offs (0-100) ----
NAME_FUZZ_FULL     = 90   
NAME_FUZZ_PARTIAL  = 75   
ADDRESS_FUZZ_FULL    = 90    
ADDRESS_EXACT_MATCH  = 100   
DOB_NEAR_PARTIAL   = 60   
EMAIL_DOMAIN_ONLY  = 5    

# ---- Auto-merge guardrails ----
DOB_IS_STRONG_ID         = False
REQUIRE_SUPPORTING_FIELD = True   # require >=1 supporting identity field too
HARD_CONFLICT_SCORE_FLOOR = 50     # hard conflict + score below this -> UNIQUE (not worth reviewing)
MIN_ACTIVE_FIELDS_FOR_MERGE = 0    # 0 = no minimum; set to 3 to block sparse pairs

FUZZY_SCORERS = {
    "surname":   "ratio",
    "givenname": "ratio",
    "address":   "token_set_ratio",
}

# ---- Blocking passes (NO standalone full DOB). Flip to False to disable. ----
BLOCKING_PASSES = {
    "B_EMAIL":              True,   # exact email
    "B_PHONE":              True,   # exact phone (pooled mobile/home/office)
    "B_DOB_SURNAME3":       True,   # full DOB + surname[:3]  (DOB always name-anchored)
    "B_SOUNDEX_YEAR":       True,   # soundex(surname) + birth year
    "B_LAST3_FIRST2_YEAR":  True,   # surname[:3] + given[:2] + birth year
    "B_POSTAL_SOUNDEX":     True,   # postal + soundex(surname)
}

# ---- Scale guard ----
MAX_BLOCK_SIZE = 100
CHUNK_SIZE     = 50_000   # scoring batch size; progress printed per chunk

# ---- Memory guard (set to ~85% of instance RAM) ----
# ml.m5.2xlarge = 32 GB -> abort at 27 GB to leave room for OS
MEM_ABORT_GB = 27

# ---- Run metadata ----
RUN_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# ---- CONFIG VALIDATION (NEW) ----
_tw = sum(FIELD_WEIGHTS.values())
if _tw != 100:
    raise ValueError(f"FIELD_WEIGHTS must sum to 100, got {_tw}.")

if not (0 < THRESHOLD_REVIEW_LOW < THRESHOLD_MERGE <= 100):
    raise ValueError(
        f"Thresholds out of order: REVIEW_LOW={THRESHOLD_REVIEW_LOW}, "
        f"MERGE={THRESHOLD_MERGE}. Need 0 < REVIEW_LOW < MERGE <= 100."
    )

if ADDRESS_EXACT_MATCH < ADDRESS_FUZZ_FULL:
    raise ValueError(
        f"ADDRESS_EXACT_MATCH ({ADDRESS_EXACT_MATCH}) must be >= "
        f"ADDRESS_FUZZ_FULL ({ADDRESS_FUZZ_FULL})."
    )

if MAX_BLOCK_SIZE < 2:
    raise ValueError(f"MAX_BLOCK_SIZE={MAX_BLOCK_SIZE} — must be >= 2.")

if CHUNK_SIZE < 100:
    raise ValueError(f"CHUNK_SIZE={CHUNK_SIZE} — must be >= 100.")

for field, scorer_name in FUZZY_SCORERS.items():
    if not hasattr(fuzz, scorer_name):
        raise ValueError(
            f"FUZZY_SCORERS['{field}'] = '{scorer_name}' — "
            f"not a valid rapidfuzz.fuzz method. "
            f"Valid: ratio, partial_ratio, token_sort_ratio, token_set_ratio, WRatio"
        )

print(f"FIELD_WEIGHTS sum = {_tw}  OK")
print(f"Thresholds: MERGE>={THRESHOLD_MERGE}, EYEBALL {THRESHOLD_REVIEW_LOW}-{THRESHOLD_MERGE-1}, UNIQUE<{THRESHOLD_REVIEW_LOW}")
print(f"ADDRESS_FUZZ_FULL={ADDRESS_FUZZ_FULL}, ADDRESS_EXACT_MATCH={ADDRESS_EXACT_MATCH}")
print(f"Blocking passes ON: {[k for k,v in BLOCKING_PASSES.items() if v]}")
print(f"MAX_BLOCK_SIZE = {MAX_BLOCK_SIZE}")
print(f"CHUNK_SIZE     = {CHUNK_SIZE:,}")
print(f"MEM_ABORT_GB   = {MEM_ABORT_GB}")
print(f"RUN_DATE = {RUN_DATE}")

FIELD_WEIGHTS sum = 100  OK
Thresholds: MERGE>=90, EYEBALL 70-89, UNIQUE<70
ADDRESS_FUZZ_FULL=90, ADDRESS_EXACT_MATCH=100
Blocking passes ON: ['B_EMAIL', 'B_PHONE', 'B_DOB_SURNAME3', 'B_SOUNDEX_YEAR', 'B_LAST3_FIRST2_YEAR', 'B_POSTAL_SOUNDEX']
MAX_BLOCK_SIZE = 100
CHUNK_SIZE     = 50,000
MEM_ABORT_GB   = 27
RUN_DATE = 2026-07-15


## CELL 3 — Load standardized data from S3

In [3]:
import os
import awswrangler as wr
wr.engine.set("python")
wr.memory_format.set("pandas")

ENVIRONMENT = os.getenv("CDCU_ENVIRONMENT", "uat")
DATA_LAKE_BUCKET = os.getenv("CDCU_DATA_LAKE_BUCKET", f"cdcu-{ENVIRONMENT}-data-lake")

# Override CDCU_STANDARDIZED_S3_URI when validating SIT legacy paths.
S3_STANDARDIZED = os.getenv(
    "CDCU_STANDARDIZED_S3_URI",
    f"s3://{DATA_LAKE_BUCKET}/standardized/merged/"
)

try:
    df = wr.s3.read_parquet(S3_STANDARDIZED, dataset=True)
except Exception as e:
    raise RuntimeError(
        f"Failed to read from S3:\n"
        f"  Path: {S3_STANDARDIZED}\n"
        f"  Error: {e}\n\n"
        f"Check:\n"
        f"  1. S3 path is correct\n"
        f"  2. IAM role has s3:GetObject permission\n"
        f"  3. Data exists at that path\n"
        f"  4. SageMaker execution role is attached"
    ) from e

if len(df) == 0:
    raise RuntimeError(
        f"S3 read returned 0 rows from {S3_STANDARDIZED}. "
        f"Check if the standardization job ran successfully."
    )

EXPECTED = ["record_id", "surname", "givenname", "dob", "address", "email",
            "mobileno", "homeno", "officeno", "postal", "event_timestamp"]

# If the table uses 'id' / 'client_no' instead of 'record_id', map it here:
if "record_id" not in df.columns:
    mapped = False
    for cand_id in ("client_no", "id"):
        if cand_id in df.columns:
            df["record_id"] = df[cand_id].astype(str)
            print(f"Mapped '{cand_id}' -> 'record_id'")
            mapped = True
            break
    if not mapped:
        raise RuntimeError(
            f"No 'record_id', 'client_no', or 'id' column found.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Cannot proceed without a unique record identifier."
        )

missing = [c for c in EXPECTED if c not in df.columns]
if missing:
    raise RuntimeError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {list(df.columns)}\n"
        f"Fix the standardization job output before running matching."
    )

print("All expected columns present.")

# ---- Pre-dedup by record_id (client_no) ----
before_dedup = len(df)
COMPLETENESS_FIELDS = ["record_id", "surname", "givenname", "dob", "address", "email", "mobileno",
                       "homeno", "officeno", "postal"]
df["_completeness"] = df[COMPLETENESS_FIELDS].notna().sum(axis=1)
df = (df.sort_values("_completeness", ascending=False)
        .drop_duplicates(subset="record_id", keep="first")
        .drop(columns="_completeness")
        .reset_index(drop=True))
after_dedup = len(df)
print(f"Loaded {before_dedup:,} rows -> pre-dedup by record_id -> {after_dedup:,} unique records")
if before_dedup > after_dedup:
    print(f"  removed {before_dedup - after_dedup:,} duplicate rows")
print(f"Memory: {format_mem_gb()}")
df.head()


Mapped 'client_no' -> 'record_id'
All expected columns present. ✓


Loaded 100,000 rows -> pre-dedup by client_no -> 85,974 unique records
  removed 14,026 duplicate rows
  (same customer, different policies — not needed for matching)
Memory: 0.7 GB


,id,client_no,surname_raw,givenname_raw,address_raw,tin,dob_raw,email_raw,gender,civil_status,mobileno_raw,homeno_raw,officeno_raw,source,inserted_at,event_timestamp,postal_raw,contract_type,standardization_timestamp,surname,surname_suffix,surname_dq_flag,givenname,givenname_suffix,givenname_dq_flag,dob,dob_dq_flag,address,address_dq_flag,email,email_valid,email_dq_flag,mobileno,mobileno_valid,mobileno_dq_flag,postal,postal_valid,postal_dq_flag,homeno,homeno_type,homeno_dq_flag,officeno,officeno_type,officeno_dq_flag,extraction_date,record_id
0,79191,00529932,BOVYYR,PNEBYVAN RTNZVAB,BLOCK 3 LOT 7 ACM WOODSTOCK HOMES ALAPAN 1 A I...,01,19710610,carolinaobille@gmail.com,F,M,09202953032,9291392,7201000,legacy,2026-06-26 01:19:34,,4103,,2026-07-15 16:21:02.466401,BOVYYR,<NA>,OK,PNEBYVAN RTNZVAB,<NA>,OK,19710610,OK,BLOCK 3 LOT 7 ACM WOODSTOCK HOMES ALAPAN 1 A I...,OK,carolinaobille@gmail.com,True,OK,+639202953032,True,OK,4103,True,OK,9291392,TELEPHONE,OK,7201000,TELEPHONE,OK,2026-07-15,00529932
1,86342,00465798,NYVCVG,SENAPVFPB Q.,"110 MAIN STREET, AEROPARK SUBDIVISION, PARANAQ...",127582938,19510508,alipitfj@yahoo.com,M,M,09165534925,87380189,8231185,legacy,2026-06-26 01:23:56,,1700,,2026-07-15 16:21:02.466401,NYVCVG,<NA>,OK,SENAPVFPB Q,<NA>,OK,19510508,OK,"110 MAIN STREET, AEROPARK SUBDIVISION, PARANAQ...",OK,alipitfj@yahoo.com,True,OK,+639165534925,True,OK,1700,True,OK,87380189,TELEPHONE,OK,8231185,TELEPHONE,OK,2026-07-15,00465798
2,81244,00774341,QVNM,EBFNYVR PNFGVYYB,"164-B VILLAROSA ROAD, BRGY. BANCAO-BANCAO, PUE...",92644618600000,19630202,roseberrysweet02@gmail.com,F,M,09676757763,4337056,09292249431,legacy,2026-06-26 01:20:49,,5300,,2026-07-15 16:21:02.466401,QVNM,<NA>,OK,EBFNYVR PNFGVYYB,<NA>,OK,19630202,OK,"164-B VILLAROSA ROAD, BRGY. BANCAO-BANCAO, PUE...",OK,roseberrysweet02@gmail.com,True,OK,+639676757763,True,OK,5300,True,OK,4337056,TELEPHONE,OK,+639292249431,MOBILE,OK,2026-07-15,00774341
3,68841,00713852,PEHM,ZNEVN VFNORY TNGPUNYVNA,"50 SALANDANAN STREET PINAGBUHATAN, PASIG CITY",19486239600000,19760216,maisabelcruz@yahoo.com,F,M,09279894681,6426605,6438441,legacy,2026-06-26 01:12:26,,1602,,2026-07-15 16:21:02.466401,PEHM,<NA>,OK,ZNEVN VFNORY TNGPUNYVNA,<NA>,OK,19760216,OK,"50 SALANDANAN STREET PINAGBUHATAN, PASIG CITY",OK,maisabelcruz@yahoo.com,True,OK,+639279894681,True,OK,1602,True,OK,6426605,TELEPHONE,OK,6438441,TELEPHONE,OK,2026-07-15,00713852
4,83023,00696320,LNC,TYRAA RQJNEQ MNPNEVNF,"AVIDA CITYFLEX TOWERS BGC, TOWER 1 UNIT 216, 7...",170815148,19730615,glenn.edward.yap@gmail.com,M,M,79029488,6203600,6876615,legacy,2026-06-26 01:21:54,,1634,,2026-07-15 16:21:02.466401,LNC,<NA>,OK,TYRAA RQJNEQ MNPNEVNF,<NA>,OK,19730615,OK,"AVIDA CITYFLEX TOWERS BGC, TOWER 1 UNIT 216, 7...",OK,glenn.edward.yap@gmail.com,True,OK,79029488,False,MOBILE_INVALID_FORMAT_OR_LENGTH,1634,True,OK,6203600,TELEPHONE,OK,6876615,TELEPHONE,OK,2026-07-15,00696320


In [4]:
## exact dataset that enters scoring — post pre-dedup,

deduped_input_df = df.copy()
deduped_input_df["run_date"] = RUN_DATE

deduped_input_df.to_csv("sit_deduped_input.csv", index=False)
print(f"sit_deduped_input.csv : {len(deduped_input_df):,} records (scoring input)")
print(f"  Columns: {list(deduped_input_df.columns)}")

## CELL 4 — Value-cleaning helpers

In [6]:

_PLACEHOLDERS = {"", "none", "nan", "null", "n/a", "na", "-", "--"}

def nz(v):
    """Normalize to a clean string or None (blank/placeholder -> None)."""
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    s = str(v).strip()
    if s.lower() in _PLACEHOLDERS:
        return None
    return s

def digits_only(v):
    """Keep digits only; None if nothing left."""
    v = nz(v)
    if v is None:
        return None
    d = re.sub(r"[^0-9]", "", v)
    return d if d else None

def digits_only_t(v):
    """digits_only but returns '' (use as an equality transform)."""
    return digits_only(v) or ""


## CELL 5 — Phone pool (cross-field mobile handling)

In [8]:

def normalize_ph_number(v):
    d = digits_only(v)
    if d is None:
        return None
    if re.fullmatch(r"09\d{9}", d):
        return d[1:]
    if re.fullmatch(r"9\d{9}", d):
        return d
    if re.fullmatch(r"639\d{9}", d):
        return d[2:]
    return d  # landline / unknown -> keep digits

def build_phone_pool(row):
    pool = set()
    for c in ("mobileno", "homeno", "officeno"):
        n = normalize_ph_number(row.get(c))
        if n:
            pool.add(n)
    return pool


## CELL 6 — Blocking keys

In [9]:

def safe_soundex(s):
    s = nz(s)
    if s is None:
        return None
    alpha = re.sub(r"[^A-Za-z]", "", s)
    if not alpha:
        return None
    try:
        return jellyfish.soundex(alpha)
    except Exception:
        return None

def dob_year(v):
    d = digits_only(v)
    return d[:4] if (d and len(d) >= 4) else None

def make_blocking_keys(row):
    """Return list of (pass_id, block_key) for one record. Skips disabled passes."""
    keys = []
    sur = nz(row.get("surname"))
    giv = nz(row.get("givenname"))
    yr  = dob_year(row.get("dob"))
    dob_full = digits_only(row.get("dob"))

    if BLOCKING_PASSES["B_EMAIL"]:
        em = nz(row.get("email"))
        if em:
            keys.append(("B_EMAIL", f"em|{em.lower()}"))

    if BLOCKING_PASSES["B_PHONE"]:
        for ph in build_phone_pool(row):
            keys.append(("B_PHONE", f"ph|{ph}"))

    # DOB is always name-anchored (no standalone full-DOB pass)
    if BLOCKING_PASSES["B_DOB_SURNAME3"] and dob_full and len(dob_full) == 8 and sur:
        s3 = re.sub(r"[^A-Z]", "", sur.upper())[:3]
        if s3:
            keys.append(("B_DOB_SURNAME3", f"dobs3|{dob_full}|{s3}"))

    if BLOCKING_PASSES["B_SOUNDEX_YEAR"]:
        sx = safe_soundex(sur)
        if sx and yr:
            keys.append(("B_SOUNDEX_YEAR", f"sxy|{sx}|{yr}"))

    if BLOCKING_PASSES["B_LAST3_FIRST2_YEAR"] and sur and giv and yr:
        s3 = re.sub(r"[^A-Z]", "", sur.upper())[:3]
        g2 = re.sub(r"[^A-Z]", "", giv.upper())[:2]
        if s3 and g2:
            keys.append(("B_LAST3_FIRST2_YEAR", f"l3f2y|{s3}|{g2}|{yr}"))

    if BLOCKING_PASSES["B_POSTAL_SOUNDEX"]:
        pc = digits_only(row.get("postal"))
        sx = safe_soundex(sur)
        if pc and sx:
            keys.append(("B_POSTAL_SOUNDEX", f"pcsx|{pc}|{sx}"))

    return keys

# preview keys for first 3 rows
for _, r in df.head(3).iterrows():
    print(r['record_id'], make_blocking_keys(r.to_dict()))

00529932 [('B_EMAIL', 'em|carolinaobille@gmail.com'), ('B_PHONE', 'ph|9291392'), ('B_PHONE', 'ph|7201000'), ('B_PHONE', 'ph|9202953032'), ('B_DOB_SURNAME3', 'dobs3|19710610|BOV'), ('B_SOUNDEX_YEAR', 'sxy|B160|1971'), ('B_LAST3_FIRST2_YEAR', 'l3f2y|BOV|PN|1971'), ('B_POSTAL_SOUNDEX', 'pcsx|4103|B160')]
00465798 [('B_EMAIL', 'em|alipitfj@yahoo.com'), ('B_PHONE', 'ph|9165534925'), ('B_PHONE', 'ph|87380189'), ('B_PHONE', 'ph|8231185'), ('B_DOB_SURNAME3', 'dobs3|19510508|NYV'), ('B_SOUNDEX_YEAR', 'sxy|N121|1951'), ('B_LAST3_FIRST2_YEAR', 'l3f2y|NYV|SE|1951'), ('B_POSTAL_SOUNDEX', 'pcsx|1700|N121')]
00774341 [('B_EMAIL', 'em|roseberrysweet02@gmail.com'), ('B_PHONE', 'ph|9676757763'), ('B_PHONE', 'ph|4337056'), ('B_PHONE', 'ph|9292249431'), ('B_DOB_SURNAME3', 'dobs3|19630202|QVN'), ('B_SOUNDEX_YEAR', 'sxy|Q150|1963'), ('B_LAST3_FIRST2_YEAR', 'l3f2y|QVN|EB|1963'), ('B_POSTAL_SOUNDEX', 'pcsx|5300|Q150')]


## CELL 7 — Candidate pair generation (self-match, vectorized)

In [10]:
def generate_candidate_pairs(frame):
    t0 = time.time()

    # Guard: empty input
    if len(frame) == 0:
        print("  WARNING: Empty DataFrame passed to generate_candidate_pairs.")
        return pd.DataFrame(columns=["idx_left", "idx_right", "blocking_rule_ids"])

    # ----- Step 1: Generate blocking keys -----
    print("Generating blocking keys...")
    t_keys = time.time()
    recs = []
    key_errors = 0

    for idx, row in frame.iterrows():
        try:
            for pass_id, key in make_blocking_keys(row.to_dict()):
                recs.append((key, idx))
        except Exception as e:
            key_errors += 1
            if key_errors <= 5:
                print(f"  WARNING: Blocking key error at idx={idx}: {e}")
            elif key_errors == 6:
                print(f"  WARNING: Suppressing further key errors (>5)...")

    bk = pd.DataFrame(recs, columns=["block_key", "rec_idx"])
    del recs
    print(f"  Keys generated: {len(bk):,} ({time.time() - t_keys:.1f}s) | mem={format_mem_gb()}")
    if key_errors > 0:
        print(f"  WARNING: {key_errors:,} rows had blocking key errors (skipped)")

    if bk.empty:
        print("  No blocking keys generated.")
        return pd.DataFrame(columns=["idx_left", "idx_right", "blocking_rule_ids"])

    # ----- Step 2: Filter blocks -----
    block_sizes = bk.groupby("block_key").size()
    oversized = block_sizes[block_sizes > MAX_BLOCK_SIZE]
    active_blocks = block_sizes[(block_sizes > 1) & (block_sizes <= MAX_BLOCK_SIZE)]

    if len(oversized) > 0:
        by_prefix = (
            oversized.index.to_series()
            .apply(lambda k: k.split("|")[0])
            .value_counts()
        )
        print(f"  Skipped {len(oversized)} oversized block(s) (max={int(block_sizes.max())}):")
        for prefix, cnt in by_prefix.items():
            print(f"    {prefix}: {cnt}")
    else:
        print(f"  [blocking] no blocks exceeded MAX_BLOCK_SIZE={MAX_BLOCK_SIZE}")

    print(f"  Active blocks (size 2-{MAX_BLOCK_SIZE}): {len(active_blocks):,}")
    print(f"  Singleton blocks (skipped):  {int((block_sizes == 1).sum()):,}")

    bk = bk[bk["block_key"].isin(active_blocks.index)]
    del block_sizes, oversized, active_blocks
    gc.collect()

    # ----- Step 3: Group + generate pairs with incremental dedup -----
    PASS_BITS = {
        "em": 1, "ph": 2, "dobs3": 4,
        "sxy": 8, "l3f2y": 16, "pcsx": 32,
    }
    BIT_TO_PASS = {
        1:  "B_EMAIL",
        2:  "B_PHONE",
        4:  "B_DOB_SURNAME3",
        8:  "B_SOUNDEX_YEAR",
        16: "B_LAST3_FIRST2_YEAR",
        32: "B_POSTAL_SOUNDEX",
    }

    grouped = bk.groupby("block_key")["rec_idx"].apply(list)
    del bk
    gc.collect()

    expected_pairs = sum(len(v) * (len(v) - 1) // 2 for v in grouped.values)
    print(f"  Expected pairs (pre-dedup): {expected_pairs:,}")
    print(f"  Generating pairs block-by-block...")

    t_pairs = time.time()
    pair_rules = {}
    raw_count = 0
    report_interval = 2_000_000

    for block_key, rec_indices in grouped.items():
        prefix = block_key.split("|")[0]
        bit = PASS_BITS.get(prefix, 0)

        for i in range(len(rec_indices)):
            for j in range(i + 1, len(rec_indices)):
                a, b = rec_indices[i], rec_indices[j]
                key = (a, b) if a < b else (b, a)
                pair_rules[key] = pair_rules.get(key, 0) | bit
                raw_count += 1

                if raw_count % report_interval == 0:
                    elapsed = time.time() - t_pairs
                    speed = raw_count / max(elapsed, 0.001)
                    pct = raw_count / max(expected_pairs, 1) * 100
                    current_mem = mem_gb()
                    print(f"    {raw_count:,}/{expected_pairs:,} ({pct:.0f}%) "
                          f"| unique: {len(pair_rules):,} "
                          f"| {speed:,.0f}/s | mem={format_mem_gb()}")

                    # Memory guard (NEW)
                    if current_mem is not None and current_mem > MEM_ABORT_GB:
                        raise MemoryError(
                            f"Memory {current_mem:.1f}GB exceeds MEM_ABORT_GB "
                            f"({MEM_ABORT_GB}GB) during pair generation. "
                            f"Generated {len(pair_rules):,} pairs before abort. "
                            f"Options: increase MEM_ABORT_GB, lower MAX_BLOCK_SIZE, "
                            f"or move to Spark."
                        )

    del grouped
    gc.collect()

    n_unique = len(pair_rules)
    print(f"  Pairs: {raw_count:,} raw -> {n_unique:,} unique "
          f"({time.time() - t_pairs:.1f}s)")

    # ----- Step 4: Convert dict to DataFrame -----
    def bitmask_to_passes(mask):
        return ",".join(sorted(
            BIT_TO_PASS[b] for b in BIT_TO_PASS if mask & b
        ))

    t_conv = time.time()
    rows = [
        (left, right, bitmask_to_passes(mask))
        for (left, right), mask in pair_rules.items()
    ]
    del pair_rules
    gc.collect()

    result = pd.DataFrame(rows, columns=["idx_left", "idx_right", "blocking_rule_ids"])
    del rows
    gc.collect()

    brute = len(frame) * (len(frame) - 1) // 2
    total_time = time.time() - t0
    print(f"  Dict->DataFrame: {time.time() - t_conv:.1f}s")
    print(f"\nCandidate pairs generated: {n_unique:,}  "
          f"(vs {brute:,} brute-force)")
    print(f"Total blocking time: {total_time:.1f}s | mem={format_mem_gb()}")

    return result

candidates = generate_candidate_pairs(df)
print(candidates.head())

Generating blocking keys...


  Keys generated: 431,784 (8.3s) | mem=0.7GB


  [blocking] no blocks exceeded MAX_BLOCK_SIZE=100
  Active blocks (size 2-100): 31,585
  Singleton blocks (skipped):  338,564


  Expected pairs (pre-dedup): 164,634
  Generating pairs block-by-block...


  Pairs: 164,634 raw -> 152,203 unique (0.2s)


  Dict->DataFrame: 0.5s

Candidate pairs generated: 152,203  (vs 3,695,721,351 brute-force)
Total blocking time: 10.6s | mem=0.7GB
   idx_left  idx_right                                  blocking_rule_ids
0     27137      44662  B_DOB_SURNAME3,B_LAST3_FIRST2_YEAR,B_SOUNDEX_YEAR
1      6032      28066  B_DOB_SURNAME3,B_EMAIL,B_LAST3_FIRST2_YEAR,B_P...
2     22258      29817  B_DOB_SURNAME3,B_LAST3_FIRST2_YEAR,B_POSTAL_SO...
3      7648      21896  B_DOB_SURNAME3,B_LAST3_FIRST2_YEAR,B_PHONE,B_P...
4     45240      46492  B_DOB_SURNAME3,B_EMAIL,B_LAST3_FIRST2_YEAR,B_P...


## CELL 8 — Per-field scoring functions

In [11]:
def _fuzz_score(a, b, scorer_name):
    """Apply the named RapidFuzz scorer; abstain (None) if either side missing."""
    a, b = nz(a), nz(b)
    if a is None or b is None:
        return None
    scorer = getattr(fuzz, scorer_name)
    return float(scorer(a.upper(), b.upper()))

def score_surname(a, b):   return _fuzz_score(a, b, FUZZY_SCORERS["surname"])
def score_givenname(a, b): return _fuzz_score(a, b, FUZZY_SCORERS["givenname"])
def score_address(a, b):   return _fuzz_score(a, b, FUZZY_SCORERS["address"])

def score_dob(a, b):
    a, b = digits_only(a), digits_only(b)
    if a is None or b is None:
        return None
    if a == b:
        return 100.0
    if len(a) == 8 and len(b) == 8 and a[:6] == b[:6]:   # same year+month
        return float(DOB_NEAR_PARTIAL)
    return 0.0

def score_email(a, b):
    a, b = nz(a), nz(b)
    if a is None or b is None:
        return None
    a, b = a.lower(), b.lower()
    if a == b:
        return 100.0
    da = a.split("@")[-1] if "@" in a else ""
    db = b.split("@")[-1] if "@" in b else ""
    if da and da == db:
        return float(EMAIL_DOMAIN_ONLY)
    return 0.0

def score_exact(a, b, transform):
    a, b = nz(a), nz(b)
    if a is None or b is None:
        return None
    return 100.0 if transform(a) == transform(b) else 0.0

def score_phone_pool(pool_a, pool_b):
    if not pool_a or not pool_b:
        return None
    return 100.0 if (pool_a & pool_b) else 0.0

print("surname DELACRUZ/DELA CRUZ:", score_surname("DELACRUZ", "DELA CRUZ"))
print("given ANA/MARIANA (stays low):", score_givenname("ANA", "MARIANA"))
print("address extra token:", score_address("123 MABINI ST", "123 MABINI ST BRGY 5"))

surname DELACRUZ/DELA CRUZ: 94.11764705882352
given ANA/MARIANA (stays low): 60.0
address extra token: 100.0


## CELL 9 — Weighted aggregation with abstain redistribution

In [13]:

def weighted_score(field_scores):
    active = {f: s for f, s in field_scores.items() if s is not None}
    if not active:
        return 0.0, {}
    total_w = sum(FIELD_WEIGHTS[f] for f in active)
    if total_w == 0:
        return 0.0, {}
    contribs, agg = {}, 0.0
    for f, s in active.items():
        w = FIELD_WEIGHTS[f] / total_w
        c = s * w
        contribs[f] = round(c, 2)
        agg += c
    return round(agg, 2), contribs

## CELL 10 — Pair evaluation: scoring + hard conflicts + classification

In [14]:
def evaluate_pair(left, right):
    pool_l = build_phone_pool(left)
    pool_r = build_phone_pool(right)

    fs = {
        "surname":   score_surname(left.get("surname"),   right.get("surname")),
        "givenname": score_givenname(left.get("givenname"), right.get("givenname")),
        "dob":       score_dob(left.get("dob"),        right.get("dob")),
        "address":   score_address(left.get("address"),right.get("address")),
        "email":     score_email(left.get("email"),    right.get("email")),
        "mobileno":  score_phone_pool(pool_l, pool_r),
        "postal":    score_exact(left.get("postal"),   right.get("postal"),   digits_only_t),
        "homeno":    score_exact(left.get("homeno"),   right.get("homeno"),   lambda x: normalize_ph_number(x) or ""),
        "officeno":  score_exact(left.get("officeno"), right.get("officeno"), lambda x: normalize_ph_number(x) or ""),
    }
    score, contribs = weighted_score(fs)

    # --- signals ---
    sur_match  = fs["surname"]   is not None and fs["surname"]   >= NAME_FUZZ_FULL
    giv_match  = fs["givenname"] is not None and fs["givenname"] >= NAME_FUZZ_FULL
    giv_diff   = fs["givenname"] is not None and fs["givenname"] <  NAME_FUZZ_PARTIAL
    name_match = sur_match and giv_match
    name_diff  = (fs["surname"] is not None and fs["surname"] < NAME_FUZZ_PARTIAL) and giv_diff
    dob_match  = fs["dob"] == 100.0
    dob_conf   = fs["dob"] == 0.0
    mob_match  = fs["mobileno"] == 100.0
    mob_conf   = fs["mobileno"] == 0.0
    email_match= fs["email"] == 100.0
    addr_match = fs["address"] is not None and fs["address"] >= ADDRESS_FUZZ_FULL   # for support
    addr_exact = fs["address"] is not None and fs["address"] >= ADDRESS_EXACT_MATCH # for household
    postal_match = fs["postal"] == 100.0


    
    has_strong_id = mob_match or email_match or (dob_match and DOB_IS_STRONG_ID)
    #supporting_field_matched = (
     #   sur_match or giv_match or dob_match or addr_match or (postal_match and sur_match)
    #)

        # need 2 or more
    support_count = sum([sur_match, giv_match, dob_match, addr_match, postal_match])
    supporting_field_matched = support_count >= 2
    if not REQUIRE_SUPPORTING_FIELD:
        supporting_field_matched = True

    # min-fields guard: if too few fields have values, do not auto-merge
    active_count = sum(1 for s in fs.values() if s is not None)
    too_few_fields = (MIN_ACTIVE_FIELDS_FOR_MERGE > 0 and
                      active_count < MIN_ACTIVE_FIELDS_FOR_MERGE)
    
    # --- hard conflict rules ---
    conflicts = []
    if mob_match and name_diff and dob_conf:
        conflicts.append("shared_mobile_diff_person")
    if name_match and dob_conf:
        conflicts.append("same_name_diff_dob")
    if addr_exact and sur_match and giv_diff:    # ← addr_exact, hindi addr_match
        conflicts.append("same_household_diff_first_name")  
    hard_conflict = len(conflicts) > 0
  

        # --- classification + reason code ---
    if hard_conflict and score >= HARD_CONFLICT_SCORE_FLOOR:                           ##eto 50 naka set
        status, reason = "EYEBALL", "hard_conflict:" + "|".join(conflicts)
    elif hard_conflict and score < HARD_CONFLICT_SCORE_FLOOR:                         ##eto 50 naka set
        status, reason = "UNIQUE", "hard_conflict_low_score:" + "|".join(conflicts)
    elif score >= THRESHOLD_MERGE and has_strong_id and supporting_field_matched and not too_few_fields:
        status, reason = "MERGE", "high_score_strong_id_plus_support"
    elif score >= THRESHOLD_MERGE and has_strong_id and too_few_fields:
        status, reason = "EYEBALL", "high_score_but_too_few_fields"
    elif score >= THRESHOLD_MERGE and has_strong_id and not supporting_field_matched:
        status, reason = "EYEBALL", "strong_id_only_no_support"
    elif score >= THRESHOLD_MERGE and not has_strong_id:
        status, reason = "EYEBALL", "high_score_no_id_anchor"
    elif score >= THRESHOLD_REVIEW_LOW:
        status, reason = "EYEBALL", "mid_score_review"
    else:
        status, reason = "UNIQUE", "low_score"

    return {
        "match_score":        score,
        "classification":     status,
        "reason_code":        reason,
        "hard_conflict_flag": "Y" if hard_conflict else "N",
        "conflict_fields":    ",".join([f for f, fl in [("dob", dob_conf), ("mobileno", mob_conf)] if fl]),
        "missing_fields":     ",".join([f for f, s in fs.items() if s is None]),
        "field_scores_json":  json.dumps(fs),
    }

# smoke test on first candidate pair
if "candidates" not in globals() or len(candidates) == 0:
    print("Run CELL 7 first (it creates `candidates`), then re-run this cell.")
else:
    _p = candidates.iloc[0]
    print(evaluate_pair(df.loc[_p['idx_left']].to_dict(),
                        df.loc[_p['idx_right']].to_dict()))

{'match_score': 82.9, 'classification': 'EYEBALL', 'reason_code': 'mid_score_review', 'hard_conflict_flag': 'N', 'conflict_fields': '', 'missing_fields': 'mobileno,postal,homeno,officeno', 'field_scores_json': '{"surname": 100.0, "givenname": 100.0, "dob": 100.0, "address": 83.51648351648352, "email": 5.0, "mobileno": null, "postal": null, "homeno": null, "officeno": null}'}


## CELL 11 — Score all pairs, classify, build the three outputs

In [15]:
def score_all_pairs(frame, cand, chunk_size=CHUNK_SIZE):
    
    # Guard: empty candidates (NEW)
    if len(cand) == 0:
        print("  No candidate pairs to score.")
        return pd.DataFrame(columns=[
            "pair_id", "record_id_left", "record_id_right", "match_score",
            "classification", "reason_code", "hard_conflict_flag",
            "conflict_fields", "missing_fields", "event_timestamp_left",
            "event_timestamp_right", "blocking_rule_ids",
            "field_scores_json", "run_date",
        ])

    total = len(cand)
    n_chunks = (total + chunk_size - 1) // chunk_size
    all_results = []
    total_errors = 0
    error_pairs = []       # first 20 error pair IDs for debugging
    t_start = time.time()
    mem_aborted = False

    print(f"Scoring {total:,} candidate pairs in {n_chunks} chunk(s) "
          f"of {chunk_size:,}...")
    print()

    for chunk_num in range(n_chunks):
        start_idx = chunk_num * chunk_size
        end_idx = min(start_idx + chunk_size, total)
        chunk = cand.iloc[start_idx:end_idx]
        t_chunk = time.time()
        chunk_errors = 0

        chunk_results = []
        for pair_num, pr in enumerate(
            chunk.itertuples(index=False), start=start_idx + 1
        ):
            try:
                left  = frame.loc[pr.idx_left].to_dict()
                right = frame.loc[pr.idx_right].to_dict()
                res = evaluate_pair(left, right)
                chunk_results.append({
                    "pair_id":              f"P{pair_num:06d}",
                    "record_id_left":       left["record_id"],
                    "record_id_right":      right["record_id"],
                    "match_score":          res["match_score"],
                    "classification":       res["classification"],
                    "reason_code":          res["reason_code"],
                    "hard_conflict_flag":   res["hard_conflict_flag"],
                    "conflict_fields":      res["conflict_fields"],
                    "missing_fields":       res["missing_fields"],
                    "event_timestamp_left": left.get("event_timestamp"),
                    "event_timestamp_right":right.get("event_timestamp"),
                    "blocking_rule_ids":    pr.blocking_rule_ids,
                    "field_scores_json":    res["field_scores_json"],
                    "run_date":             RUN_DATE,
                })
            except Exception as e:
                chunk_errors += 1
                total_errors += 1
                if len(error_pairs) < 20:
                    error_pairs.append({
                        "pair_num": pair_num,
                        "idx_left": pr.idx_left,
                        "idx_right": pr.idx_right,
                        "error": str(e),
                    })

        chunk_df = pd.DataFrame(chunk_results)
        all_results.append(chunk_df)

        elapsed = time.time() - t_chunk
        total_elapsed = time.time() - t_start
        pairs_done = end_idx
        pairs_left = total - pairs_done
        speed = pairs_done / max(total_elapsed, 0.001)
        eta = pairs_left / max(speed, 1)
        err_str = f" | ERRORS: {chunk_errors}" if chunk_errors > 0 else ""

        print(f"  chunk {chunk_num+1}/{n_chunks}: pairs {start_idx+1:,}-"
              f"{end_idx:,} ({elapsed:.1f}s) | total {pairs_done:,}/"
              f"{total:,} | ETA {eta:.0f}s{err_str}")

        # Memory guard: return what we have so far (NEW)
        current_mem = mem_gb()
        if current_mem is not None and current_mem > MEM_ABORT_GB:
            print(f"\n  WARNING: Memory {current_mem:.1f}GB exceeds "
                  f"MEM_ABORT_GB ({MEM_ABORT_GB}GB).")
            print(f"  Returning {pairs_done:,} scored pairs (out of {total:,}).")
            mem_aborted = True
            break

    pairs_df = pd.concat(all_results, ignore_index=True)
    total_time = time.time() - t_start
    print(f"\nDone: {len(pairs_df):,} pairs scored in {total_time:.1f}s "
          f"({len(pairs_df) / max(total_time, 0.001):,.0f} pairs/sec)")

    if total_errors > 0:
        print(f"\n  WARNING: {total_errors:,} pairs failed during scoring (skipped).")
        print(f"  First errors:")
        for ep in error_pairs[:5]:
            print(f"    pair {ep['pair_num']}: idx_left={ep['idx_left']}, "
                  f"idx_right={ep['idx_right']} -> {ep['error']}")

    if mem_aborted:
        print(f"\n  PARTIAL RESULTS: scoring stopped early due to memory.")

    return pairs_df

pairs_df = score_all_pairs(df, candidates)

# pair-level buckets
merge_df   = pairs_df[pairs_df["classification"] == "MERGE"].reset_index(drop=True)
eyeball_df = pairs_df[pairs_df["classification"] == "EYEBALL"].reset_index(drop=True)

# record-level UNIQUE = records never in a MERGE/EYEBALL pair
matched_ids = (
    set(merge_df["record_id_left"]) | set(merge_df["record_id_right"])
    | set(eyeball_df["record_id_left"]) | set(eyeball_df["record_id_right"])
)
unique_df = df[~df["record_id"].isin(matched_ids)].copy()
unique_df = unique_df[
    ["record_id", "surname", "givenname", "dob", "email",
     "mobileno", "event_timestamp"]
].reset_index(drop=True)
unique_df["classification"] = "UNIQUE"
unique_df["reason_code"]    = "no_duplicate_found"
unique_df["run_date"]       = RUN_DATE

print()
print("=" * 46)
print(f"Input records         : {len(df):,}")
print(f"MERGE pairs           : {len(merge_df):,}")
print(f"EYEBALL pairs         : {len(eyeball_df):,}")
print(f"Records in MERGE      : {len(set(merge_df['record_id_left']) | set(merge_df['record_id_right'])):,}")
print(f"Records in EYEBALL    : {len(set(eyeball_df['record_id_left']) | set(eyeball_df['record_id_right'])):,}")
print(f"UNIQUE records        : {len(unique_df):,}")
print("=" * 46)
print("\nSample MERGE pairs:")
if len(merge_df):
    print(merge_df[["pair_id","record_id_left","record_id_right","match_score","reason_code"]].head(5).to_string(index=False))
else:
    print("  (none)")
print("\nSample EYEBALL pairs:")
if len(eyeball_df):
    print(eyeball_df[["pair_id","record_id_left","record_id_right","match_score","reason_code"]].head(5).to_string(index=False))
else:
    print("  (none)")
print(f"\nMemory: {format_mem_gb()}")

  chunk 1/4: pairs 1-50,000 (34.0s) | total 50,000/152,203 | ETA 69s


  chunk 2/4: pairs 50,001-100,000 (34.7s) | total 100,000/152,203 | ETA 36s


  chunk 3/4: pairs 100,001-150,000 (33.5s) | total 150,000/152,203 | ETA 1s


  chunk 4/4: pairs 150,001-152,203 (1.4s) | total 152,203/152,203 | ETA 0s

Done: 152,203 pairs scored in 103.5s (1,470 pairs/sec)



Input records         : 85,974
MERGE pairs           : 587
EYEBALL pairs         : 2,645
Records in MERGE      : 1,091
Records in EYEBALL    : 3,955
UNIQUE records        : 81,103

Sample MERGE pairs:
pair_id record_id_left record_id_right  match_score                       reason_code
P000002       00851789        00068907        99.00 high_score_strong_id_plus_support
P000018       00141819        00820578        95.34 high_score_strong_id_plus_support
P000022       00921331        00982122        92.41 high_score_strong_id_plus_support
P000029       00637957        00922185        90.53 high_score_strong_id_plus_support
P000056       00054224        00185919        94.75 high_score_strong_id_plus_support

Sample EYEBALL pairs:
pair_id record_id_left record_id_right  match_score      reason_code
P000001       00931575        00544078        82.90 mid_score_review
P000004       00070367        00023911        76.32 mid_score_review
P000005       00157735        00299837        87.12 

## CELL 11b — Reason-code breakdown (reporting)

In [16]:
dup_counts = df.groupby("record_id").size()
dups = dup_counts[dup_counts > 1]
print(f"client_no na lumalabas ng higit sa 1 beses: {len(dups)}")
print(f"Total rows na duplicate: {dups.sum()}")
print(dups.sort_values(ascending=False).head(10))

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 7.87 μs
client_no na lumalabas ng higit sa 1 beses: 0
Total rows na duplicate: 0
Series([], dtype: int64)


In [17]:

print("PAIR-LEVEL reason codes")
print("-" * 46)
print(pairs_df.groupby(["classification","reason_code"]).size().to_string())

print("\nREPORTING VIEW (pair vs record counts — keep separate)")
print("-" * 46)
rec_merge   = set(merge_df['record_id_left'])   | set(merge_df['record_id_right'])
rec_eyeball = set(eyeball_df['record_id_left']) | set(eyeball_df['record_id_right'])
report = pd.DataFrame([
    ["Total input records",        len(df),            "all standardized rows processed"],
    ["MERGE pairs",                len(merge_df),      "high-confidence duplicate relationships"],
    ["EYEBALL pairs",              len(eyeball_df),    "borderline relationships for review"],
    ["Records involved in MERGE",  len(rec_merge),     "rows in >=1 merge candidate pair"],
    ["Records involved in EYEBALL",len(rec_eyeball),   "rows in >=1 review pair"],
    ["UNIQUE records",             len(unique_df),     "rows with no match found this run"],
], columns=["metric", "count", "business_interpretation"])
print(report.to_string(index=False))

PAIR-LEVEL reason codes
----------------------------------------------
classification  reason_code                                           
EYEBALL         hard_conflict:same_household_diff_first_name                1237
                hard_conflict:same_name_diff_dob                              56
                hard_conflict:shared_mobile_diff_person                      251
                mid_score_review                                            1085
                strong_id_only_no_support                                     16
MERGE           high_score_strong_id_plus_support                            587
UNIQUE          hard_conflict_low_score:same_household_diff_first_name       237
                hard_conflict_low_score:same_name_diff_dob                     3
                hard_conflict_low_score:shared_mobile_diff_person           5850
                low_score                                                 142881

REPORTING VIEW (pair vs record counts — keep se

## CELL 11d — Suggested survivor for MERGE pairs (no deletion)

In [19]:
def suggest_survivor(merge_pairs, frame):
    """For each MERGE pair, suggest the latest record_id by event_timestamp. No deletion."""
    if merge_pairs.empty:
        return pd.DataFrame(columns=["pair_id","record_id_left","record_id_right",
                                     "suggested_survivor","survivor_reason"])
    # Dedup by record_id: if a client_no appears multiple times (source duplicates),
    # keep the latest timestamp per record_id so the lookup returns ONE value.
    _ts = frame[["record_id","event_timestamp"]].copy()
    _ts["event_timestamp"] = pd.to_datetime(_ts["event_timestamp"], errors="coerce")
    _ts = _ts.sort_values("event_timestamp", ascending=False).drop_duplicates(subset="record_id", keep="first")
    ts = _ts.set_index("record_id")["event_timestamp"]
    out = []
    for pr in merge_pairs.itertuples(index=False):
        tl, tr = ts.get(pr.record_id_left), ts.get(pr.record_id_right)
        if pd.notna(tl) and pd.notna(tr):
            survivor = pr.record_id_left if tl >= tr else pr.record_id_right
        elif pd.notna(tl):
            survivor = pr.record_id_left
        else:
            survivor = pr.record_id_right
        out.append({"pair_id": pr.pair_id,
                    "record_id_left": pr.record_id_left,
                    "record_id_right": pr.record_id_right,
                    "suggested_survivor": survivor,
                    "survivor_reason": "latest_event_timestamp"})
    return pd.DataFrame(out)

survivor_df = suggest_survivor(merge_df, df)
print(f"Survivor suggestions for {len(survivor_df)} MERGE pairs (nothing deleted):")
print(survivor_df.head(8).to_string(index=False) if len(survivor_df) else "(no MERGE pairs)")


Survivor suggestions for 587 MERGE pairs (nothing deleted):
pair_id record_id_left record_id_right suggested_survivor        survivor_reason
P000002       00851789        00068907           00068907 latest_event_timestamp
P000018       00141819        00820578           00820578 latest_event_timestamp
P000022       00921331        00982122           00982122 latest_event_timestamp
P000029       00637957        00922185           00922185 latest_event_timestamp
P000056       00054224        00185919           00185919 latest_event_timestamp
P000060       00983756        01005276           01005276 latest_event_timestamp
P000063       00150803        00150789           00150789 latest_event_timestamp
P000064       00479543        00334293           00334293 latest_event_timestamp


## CELL 11f — Graph clustering (pairs to clusters)


In [21]:
# ---- Graph clustering config ----
CLUSTER_INCLUDE_EYEBALL    = False
CLUSTER_EYEBALL_MIN_SCORE  = 85
CLUSTER_MAX_SIZE_WARN      = 30

def build_clusters(merge_pairs, eyeball_pairs=None):
    """Union-find connected components over MERGE (and optional strong EYEBALL) pairs.
    Returns a DataFrame: record_id -> cluster_id, plus cluster size."""
    edges = list(zip(merge_pairs["record_id_left"], merge_pairs["record_id_right"]))
    if CLUSTER_INCLUDE_EYEBALL and eyeball_pairs is not None and len(eyeball_pairs):
        strong = eyeball_pairs[eyeball_pairs["match_score"] >= CLUSTER_EYEBALL_MIN_SCORE]
        edges += list(zip(strong["record_id_left"], strong["record_id_right"]))

    # union-find
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for a, b in edges:
        union(a, b)

    # assign stable cluster ids
    roots = {}
    rows = []
    for node in list(parent.keys()):
        r = find(node)
        if r not in roots:
            roots[r] = f"CL{len(roots)+1:06d}"
        rows.append({"record_id": node, "cluster_id": roots[r]})
    clusters = pd.DataFrame(rows)
    if clusters.empty:
        return clusters.assign(cluster_size=pd.Series(dtype=int))
    sizes = clusters.groupby("cluster_id").size().rename("cluster_size")
    return clusters.merge(sizes, on="cluster_id")

# Guard: always define clusters_df (NEW)
if len(merge_df) > 0:
    clusters_df = build_clusters(merge_df, eyeball_df)
else:
    clusters_df = pd.DataFrame(columns=["record_id", "cluster_id", "cluster_size"])

if len(clusters_df):
    n_clusters = clusters_df["cluster_id"].nunique()
    n_records  = clusters_df["record_id"].nunique()
    size_dist  = clusters_df.drop_duplicates("cluster_id")["cluster_size"]
    print(f"Records in clusters : {n_records:,}")
    print(f"Clusters formed     : {n_clusters:,}")
    print(f"Avg cluster size    : {size_dist.mean():.2f}")
    print(f"Largest cluster     : {int(size_dist.max())} records")
    big = size_dist[size_dist >= CLUSTER_MAX_SIZE_WARN]
    if len(big):
        print(f"\n  WARNING: {len(big)} cluster(s) >= {CLUSTER_MAX_SIZE_WARN} records.")
        print( "  A very large cluster usually means a junk edge slipped through")
        print( "  (e.g. a shared phone). Check JUNK_PHONES / MAX_BLOCK_SIZE.")
    print("\nCluster size distribution:")
    print(size_dist.value_counts().sort_index().head(10).to_string())
    print("\nSample clusters (size >= 2):")
    multi = clusters_df[clusters_df["cluster_size"] >= 2].sort_values(["cluster_size","cluster_id"], ascending=[False, True])
    print(multi.head(12).to_string(index=False))
else:
    print("No clusters (no MERGE pairs).")


Records in clusters : 1,091
Clusters formed     : 534
Avg cluster size    : 2.04
Largest cluster     : 6 records

Cluster size distribution:
cluster_size
2    517
3     13
4      3
6      1

Sample clusters (size >= 2):
record_id cluster_id  cluster_size
    13683   CL000119             6
     9892   CL000119             6
     9736   CL000119             6
    12044   CL000119             6
    21467   CL000119             6
    15028   CL000119             6
 00851787   CL000053             4
 00902583   CL000053             4
 00463014   CL000053             4
 00010535   CL000053             4
    48440   CL000157             4
    20019   CL000157             4


## CELL 11e — Export buckets to CSV (SIT convenience)

In [19]:
merge_df.to_csv("sit_merge_pairs.csv", index=False)
eyeball_df.to_csv("sit_eyeball_pairs.csv", index=False)
unique_df.to_csv("sit_unique_records.csv", index=False)
survivor_df.to_csv("sit_merge_survivors.csv", index=False)
if "clusters_df" in globals() and len(clusters_df):
    clusters_df.to_csv("sit_clusters.csv", index=False)
print("Wrote: sit_merge_pairs.csv, sit_eyeball_pairs.csv, sit_unique_records.csv,")
print("       sit_merge_survivors.csv, sit_clusters.csv")

## CELL 11g — Reviewer-ready output (pairs + record details)

In [20]:
# ---- Build reviewer-ready output: pairs + record details side by side ----
REVIEW_FIELDS = ["record_id", "surname", "givenname", "dob", "address",
                 "email", "mobileno", "homeno", "officeno", "postal", "event_timestamp"]

def build_review_output(pairs, source_df):
    """Join pair results with full record details from both left and right sides."""
    if pairs.empty:
        return pd.DataFrame()

    # Prepare left and right lookup tables
    left_cols  = {f: f"{f}_left"  for f in REVIEW_FIELDS}
    right_cols = {f: f"{f}_right" for f in REVIEW_FIELDS}

    src = source_df[REVIEW_FIELDS].copy()

    # Join left side
    merged = pairs.merge(
        src.rename(columns=left_cols),
        left_on="record_id_left", right_on="record_id_left", how="left"
    )
    # Join right side
    merged = merged.merge(
        src.rename(columns=right_cols),
        left_on="record_id_right", right_on="record_id_right", how="left"
    )

    # Reorder columns: pair info first, then left fields, then right fields
    pair_cols = ["pair_id", "match_score", "classification", "reason_code",
                 "hard_conflict_flag", "blocking_rule_ids"]
    pair_cols = [c for c in pair_cols if c in merged.columns]

    left_detail  = [f"{f}_left"  for f in REVIEW_FIELDS]
    right_detail = [f"{f}_right" for f in REVIEW_FIELDS]
    other_cols   = [c for c in merged.columns if c not in pair_cols + left_detail + right_detail]

    final_order = pair_cols + left_detail + right_detail + other_cols
    final_order = [c for c in final_order if c in merged.columns]
    return merged[final_order]

# Build for MERGE and EYEBALL
merge_review  = build_review_output(merge_df, df)
eyeball_review = build_review_output(eyeball_df, df)

# Save locally
merge_review.to_csv("sit_merge_review.csv", index=False)
eyeball_review.to_csv("sit_eyeball_review.csv", index=False)

print(f"sit_merge_review.csv   : {len(merge_review):,} pairs with full record details")
print(f"sit_eyeball_review.csv : {len(eyeball_review):,} pairs with full record details")
print()
print("Columns per row:")
print(f"  Pair info: pair_id, match_score, classification, reason_code, ...")
print(f"  Left side: surname_left, givenname_left, dob_left, address_left, ...")
print(f"  Right side: surname_right, givenname_right, dob_right, address_right, ...")
print()
if len(merge_review):
    print("Sample MERGE review (first 3 pairs, key fields):")
    sample_cols = ["pair_id","match_score","reason_code",
                   "surname_left","givenname_left","dob_left","mobileno_left",
                   "surname_right","givenname_right","dob_right","mobileno_right"]
    sample_cols = [c for c in sample_cols if c in merge_review.columns]
    print(merge_review[sample_cols].head(3).to_string(index=False))

## CELL 14 — Write the outputs to S3

In [22]:
#with error handling

OUTPUT_BASE = os.getenv(
    "CDCU_MATCHING_OUTPUT_S3_URI",
    f"s3://{DATA_LAKE_BUCKET}/processed/matching"
).rstrip("/")
CSV_OUTPUT_BASE = os.getenv(
    "CDCU_MATCHING_CSV_OUTPUT_S3_URI",
    f"s3://{DATA_LAKE_BUCKET}/processed/matching_csv"
).rstrip("/")
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = os.getenv("CDCU_MATCHING_RUN_ID", f"{RUN_TIMESTAMP}-{uuid.uuid4().hex[:8]}")

_survivor_df = survivor_df if "survivor_df" in globals() else pd.DataFrame()
_clusters_df = clusters_df if "clusters_df" in globals() else pd.DataFrame()

OUT = {
    "merge": (
        merge_df,
        f"{OUTPUT_BASE}/merge_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
    "eyeball": (
        eyeball_df,
        f"{OUTPUT_BASE}/eyeball_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
    "unique": (
        unique_df,
        f"{OUTPUT_BASE}/unique_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
    "merge_survivors": (
        _survivor_df,
        f"{OUTPUT_BASE}/merge_survivors_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
    "deduped_input": (
        deduped_input_df,
        f"{OUTPUT_BASE}/deduped_input_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
    "clusters": (
        _clusters_df,
        f"{OUTPUT_BASE}/clusters_sagemaker/run_date={RUN_DATE}/run_id={RUN_ID}/",
    ),
}

write_errors = []
csv_out = {
    "merge_pairs": merge_df,
    "eyeball_pairs": eyeball_df,
    "unique_records": unique_df,
    "merge_survivors": _survivor_df,
    "deduped_input": deduped_input_df,
    "clusters": _clusters_df,
    "merge_review": merge_review,
    "eyeball_review": eyeball_review,
}
csv_write_errors = []

for name, (frame, path) in OUT.items():
    if len(frame) == 0:
        print(f"{name:16s} -> SKIPPED (0 rows)")
        continue

    try:
        wr.s3.to_parquet(
            df=frame,
            path=path,
            dataset=True,
            mode="append",
            compression="snappy",
        )
        print(f"{name:16s} -> {path} ({len(frame):,} rows written)")
    except Exception as e:
        write_errors.append((name, str(e)))
        print(f"{name:16s} -> FAILED: {e}")

for name, frame in csv_out.items():
    if len(frame) == 0:
        print(f"{name:16s} CSV -> SKIPPED (0 rows)")
        continue

    csv_path = f"{CSV_OUTPUT_BASE}/{name}/run_date={RUN_DATE}/run_id={RUN_ID}/{name}.csv"
    try:
        wr.s3.to_csv(
            df=frame,
            path=csv_path,
            index=False,
        )
        print(f"{name:16s} CSV -> {csv_path} ({len(frame):,} rows written)")
    except Exception as e:
        csv_write_errors.append((name, str(e)))
        print(f"{name:16s} CSV -> FAILED: {e}")

if write_errors:
    print(f"\nWARNING: {len(write_errors)} S3 write(s) failed:")
    for wname, werr in write_errors:
        print(f"    {wname}: {werr}")
    print(f"\nCheck IAM permissions and S3 path: {OUTPUT_BASE}")
else:
    print("\nAll outputs written successfully.")

if csv_write_errors:
    print(f"\nWARNING: {len(csv_write_errors)} CSV S3 write(s) failed:")
    for wname, werr in csv_write_errors:
        print(f"    {wname}: {werr}")
    print(f"\nParquet outputs are unaffected. Check IAM permissions and S3 path: {CSV_OUTPUT_BASE}")
else:
    print("CSV review outputs written successfully.")

print(f"\nrun_date = {RUN_DATE}")
print(f"run_id   = {RUN_ID}")
print(f"Final memory: {format_mem_gb()}")
print("\nDone.")
